# Dataset Wei
## 

In [ ]:
# import all needed R packages
library(ChAMP)
library(ChAMPdata)
library(ggplot2)
library(stringr)
library(ggpubr)
library(RColorBrewer)
library(colorspace)
library(tidyr)
library(tibble)
library(dplyr)
library(IlluminaHumanMethylation450kanno.ilmn12.hg19)
source("custom_functions.R")
print("Laden der Bibliotheken erfolgreich abgeschlossen")

In [ ]:
# set the location of the data directory
# the .idat fluorescence files and the samplesheet.csv have to be located inside the directory and it's subdirectories
getwd()
wei_dir <- "/home/jovyan/datasets/data/Wei"
print("Setzen des Datenverzeichnisses erfolgreich abgeschlossen")

In [ ]:
# Load in the data
myLoad <- champ.load(directory = wei_dir,
                     method="ChAMP", # replaces old loading method using minfi
                     methValue="B", # wether to calculate beta- or M-values
                     autoimpute=TRUE, # if values are missing uses the 3 most similar probes and uses the mean of their values for the missing value 
                     filterDetP=TRUE, # filter single probes, whose methylation signal vs the background signal of the slide is not significant (detPcut)
                     ProbeCutoff=0, # remove all probes with higher missing-value-ratio
                     SampleCutoff=0.1, # remove a full sample if the failed probe ratio (based on p value) is higher
                     detPcut=0.01, # significance p-value cutoff for filterDetP
                     filterBeads=TRUE, # filter out probes if the fraction of samples with a beadcount < 3 is higher than beadCutoff
                     beadCutoff=0.05, # acceptable fraction of samples with a beadcount < 3
                     filterNoCG=TRUE, # wether to remove non-cg probes
                     filterSNPs=TRUE, # wether to remove probes that fallnear a SNP (as defined in Nordlund et al. https://link.springer.com/article/10.1186/s13059-021-02529-2) 
                     population=NULL, # can be assigned to specific population according to www.internationalgenome.org/category/population/
                     filterMultiHit=TRUE, # wether to remove probes that align to multiple genomic locations (also according to Nordlund et al.)
                     filterXY=TRUE, # wether to remove probes on x and y chromosomes
                     force=FALSE, # minfi specific parameter
                     arraytype="450K") # microarray type (can be one of "450K" "EPICv1" or "EPICv2") 

champ.QC(beta = myLoad$beta, # beta values stored in champ.load output
         pheno=myLoad$pd$Sample_Group, # what samplesheet column is your phenotype (where do u expect the major difference between your samples)?
         resultsDir="/home/jovyan/CHAMP_QCimages/") # the plots will be saved in the directory this notebook file is located in


In [ ]:
# for two of the normalization methods we need special input, which we generate below
targets <- read.metharray.sheet(wei_dir)
rgset <- read.metharray.exp(targets = targets, recursive = TRUE)
mset <- preprocessRaw(rgset)
print("Laden von mset und rgset für SWAN erfolgreich abgeschlossen")

#### Peak-based correction normalization (PBC)

In [ ]:
# start with PBC normalization
myNormPBC <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "PBC")
champ.QC(beta = myNormPBC, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/PBC/")


#### Beta-mixture quantile normalization (BMIQ)

In [ ]:
# BMIQ normalization
myNormBMIQ <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "BMIQ")
champ.QC(beta = myNormBMIQ, pheno=myLoad$pd$Sample_Group, resultsDir="/home/jovyan/CHAMP_QCimages/BMIQ/")

#### Functional Normalization (FunNorm)

In [ ]:
myNormFunNorm <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "FunctionalNormalization", rgSet = rgset)
champ.QC(beta = myNormFunNorm, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/FunNorm/")

#### Subset-quantiles within microarray normalization (SWAN)

In [ ]:
myNormSWAN <- champ.norm(arraytype = "450K", method = "SWAN", rgSet=rgset,mset = mset,beta = NULL)
colnames(myNormSWAN) <- targets$Sample_Name
pheno <- targets$Sample_Group         
names(pheno) <- targets$Sample_Name    # Echte Namen
pheno <- pheno[colnames(myNormSWAN)]   # Reihenfolge anpassen
champ.QC(beta = myNormSWAN, pheno = myLoad$pd$Sample_Group, resultsDir = "/home/jovyan/CHAMP_QCimages/SWAN/")

In [ ]:
# save best looking normalization result
myNorm <- 

In [ ]:
champ.SVD(beta=myNorm, pd=myLoad$pd, resultsDir="/home/jovyan/CHAMP_SVDimages/")
SVD_custom(beta=myNorm, filename = "/home/jovyan/CHAMP_SVDimages/SVD_Wei.pdf")

In [ ]:
DMPs <- champ.DMP(beta = myNorm,pheno=myLoad$pd$Sample_Group, adjPVal = 0.05, arraytype = "450K")
DMP.GUI(DMP=DMPs[[1]],beta=myNorm,pheno=myLoad$pd$Sample_Group)

In [ ]:
nrow(DMPs$Tumor_to_Normal)

In [ ]:
DMRs <- champ.DMR(beta=myNorm,pheno=myLoad$pd$Sample_Group,method="Bumphunter", arraytype="450K")
DMR.GUI(DMR=DMRs, arraytype="450K")

In [ ]:
nrow(DMRs$BumphunterDMR)
head(DMRs$BumphunterDMR)

In [ ]:
anno <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)

DMRs_filt <- filter_dmrs(DMRs$BumphunterDMR, pval = 0.05, fwer = 0.05,min_abs_value = 0.5)
DMRs_anno <- annotateDMRs(DMRs_filt, anno = anno, myNorm = myNorm)
DMRs_head <- DMRs_anno[1:10, ]
DMRs_tail <- DMRs_anno[(nrow(DMRs_annot)-9):nrow(DMRs_annot), ]

In [ ]:
makeHeatmap_genes_list_cluster(pdfname = "/home/jovyan/Heatmap_images/heatmap_Wei_genelist.pdf",
                            anno = anno,
                            genelist = c("NPY", "RASSF1", "CLDN10"),
                            myNorm = myNorm,
                            targets = targets,
                            pd_col = "Sample_Group",
                            compare_values = c("Tumor", "Normal"))

makeHeatmap_dmr(pdfname = "heatmap_Wei_dmrs_hypermethylated.pdf",
                dmrs = DMRs_head,
                anno = anno,
                myNorm = myNorm,
                targets = targets,
                pd_col = "Sample_Group",
                compare_values = c("Tumor", "Normal"))

makeHeatmap_dmr(pdfname = "heatmap_Wei_dmrs_hypomethylated.pdf",
                dmrs = DMRs_tail,
                anno = anno,
                myNorm = myNorm,
                targets = targets,
                pd_col = "Sample_Group",
                compare_values = c("Tumor", "Normal"))

In [ ]:
DMRs_head

In [ ]:
DMRs_tail